<a href="https://colab.research.google.com/github/Susan-sapkota-98/Nepali-asr-evaluation/blob/main/08_error_sorter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [jiwer]


In [3]:
from pathlib import Path
import jiwer
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
folder = Path("/content/drive/MyDrive/nepali-asr-evaluation/rating_200")

Mounted at /content/drive


In [4]:
results = pd.read_csv(folder / "scores_200.csv", dtype={"file_id": str}, keep_default_na=False)
print(len(results), "rows")
results.head(3)

200 rows


,file_id,correct_text,model_text,wer,cer
0,0be7aaf743,वुड्सको प्रदर्शन प्रभावशाली,हुटस्कव प्रदर्शन प्रभाव शाली,1.000000,0.222222
1,0a85921e16,मन्दिरमा तीनवटा हिन्दू,मन्दिरमा तिनोटा हिन्दु,0.666667,0.136364
2,06bc69472f,पानीको आसपास बस्छ,पानीको आसपास बस्छ,0.000000,0.000000


In [5]:
digit_map = str.maketrans("०१२३४५६७८९", "0123456789")
vowel_signs = "ािीुूृेैोौ्ंँः"


def remove_punctuation(text):
    for mark in "।,.?!;:\"'":
        text = text.replace(mark, " ")
    return " ".join(text.split())


def has_latin_letters(text):
    return any("a" <= letter.lower() <= "z" for letter in text)


def has_digit(text):
    return any(letter.isdigit() for letter in text)


def has_junk(text):
    # a real word cannot start with a vowel sign
    for word in text.split():
        if word[0] in vowel_signs:
            return True
    return False

In [6]:
def find_error_type(correct, model):
    if correct == model:
        return "exact"

    a = remove_punctuation(correct)
    b = remove_punctuation(model)
    if a == b:
        return "punctuation_only"
    if a.replace(" ", "") == b.replace(" ", ""):
        return "spacing_only"

    if has_junk(model):
        return "junk"
    if has_latin_letters(model):
        return "latin_mix"

    a_digits = a.translate(digit_map)
    b_digits = b.translate(digit_map)
    if a_digits.replace(" ", "") == b_digits.replace(" ", ""):
        return "digit_script_only"
    if has_digit(a_digits) or has_digit(b_digits):
        return "number_wrong"

    if jiwer.cer(a, b) <= 0.15:
        return "small_difference"
    return "big_difference"

In [8]:
print(find_error_type("सबैकुरा सिकाउँथे", "सबै कुरा सिकाउँथे"))
print(find_error_type("आफ्नो पुस्तक २०३२", "आफ्नो पुस्तक 2032"))
print(find_error_type("कृपया यहाँ", "कृपeya yaha"))
print(find_error_type("गर्न छोड्नु।", "गर्न छोड्नु"))

spacing_only
digit_script_only
latin_mix
punctuation_only


In [9]:
results["error_type"] = [find_error_type(c, m) for c, m in zip(results["correct_text"], results["model_text"])]
results["error_type"].value_counts()

,count
error_type,
small_difference,58
exact,55
big_difference,27
latin_mix,25
number_wrong,16
spacing_only,11
junk,7
digit_script_only,1


In [10]:
for type_name in sorted(results["error_type"].unique()):
    print("----", type_name)
    print(results[results["error_type"] == type_name][["correct_text", "model_text"]].head(3).to_string())

---- big_difference
                    correct_text                    model_text
0    वुड्सको प्रदर्शन प्रभावशाली  हुटस्कव प्रदर्शन प्रभाव शाली
23  शरद्चन्द्र शर्मा भट्टर्राईको      सरस चन्द्र सरमा भटरयाएको
39                  संवाद लेखनको                 समुवाद लेखनको
---- digit_script_only
      correct_text      model_text
74  १९३४ को जनावरी  1934 को जनावरी
---- exact
        correct_text         model_text
2  पानीको आसपास बस्छ  पानीको आसपास बस्छ
3       असामयिक करार       असामयिक करार
6   पनि गर्दै त्यसमा   पनि गर्दै त्यसमा
---- junk
               correct_text                                  model_text
26  इस्लामिया उच्च शिक्षाको  iस्लामिया उच्च शिक्षा को    ा ााा ा ा ा  ा
53             स्वस्थ टाउको                           स्ास्त्य ट ा उ का
57          १६ औँ शताब्दीमा                             16 ौँ शताब्दीमा
---- latin_mix
                 correct_text                       model_text
11      गाड्नुका साथै हाँगामा              गाडnुका साथे आँगामा
13  विपरीत अस्ट्रेलिया आर

In [11]:
group_of = {
    "exact": "exact",
    "punctuation_only": "harmless",
    "spacing_only": "harmless",
    "digit_script_only": "harmless",
    "small_difference": "needs a human",
    "junk": "real mistake",
    "latin_mix": "real mistake",
    "number_wrong": "real mistake",
    "big_difference": "real mistake",
}
results["group"] = results["error_type"].map(group_of)
results["group"].value_counts()

,count
group,
real mistake,75
needs a human,58
exact,55
harmless,12


In [12]:
results.groupby("error_type")[["wer", "cer"]].mean().round(2)

,wer,cer
error_type,,
big_difference,0.84,0.24
digit_script_only,0.33,0.29
exact,0.00,0.00
junk,1.60,0.55
latin_mix,1.54,0.77
number_wrong,0.61,0.37
small_difference,0.51,0.09
spacing_only,0.79,0.06


In [13]:
results["word_errors"] = results["wer"] * results["correct_text"].str.split().str.len()

share = results.groupby("group")["word_errors"].sum() / results["word_errors"].sum()
print((share * 100).round(1))

group
exact             0.0
harmless          6.9
needs a human    26.0
real mistake     67.1
Name: word_errors, dtype: float64


In [14]:
results.to_csv(folder / "sorted_200.csv", index=False)
print("saved")

saved
